# Saliency Robustness to Perturbation

## 1. Introduction
In highly regulated environments (e.g., healthcare, finance), it is not enough for an AI model to just be accurate; its *explanations* must also be reliable. 

This notebook demonstrates a critical XAI auditing procedure: **Explanation Robustness**. We test how the addition of minor, visually imperceptible noise to an image affects the model's "reasoning." Even if the model correctly predicts the noisy image, a drastic shift in the saliency map indicates that the explanation is fragile and untrustworthy. We quantify this using `saliencytools`' SSIM and CC metrics.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
import sys
import os

# Add parent directory to path to import saliencytools
sys.path.append(os.path.abspath('..'))
import saliencytools.maskcompare as mc

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 2. Model Definition
We use a standard CNN to ensure the results are easily reproducible.

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.fc1 = nn.Linear(32 * 7 * 7, 64)
        self.fc2 = nn.Linear(64, 10)

    def forward(self, x):
        x = F.relu(F.max_pool2d(self.conv1(x), 2))
        x = F.relu(F.max_pool2d(self.conv2(x), 2))
        x = x.view(-1, 32 * 7 * 7)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

def get_saliency(model, x, target_class):
    model.eval()
    x_input = x.clone().detach().to(device).requires_grad_(True)
    if len(x_input.shape) == 3: x_input = x_input.unsqueeze(0)
    output = model(x_input)
    output[0, target_class].backward()
    saliency = x_input.grad.data.abs().squeeze().detach().cpu().numpy()
    return (saliency - saliency.min()) / (saliency.max() - saliency.min() + 1e-8)

## 3. The Perturbation Experiment
1. **Baseline Extraction**: We compute the saliency map for a clean, unmodified image.
2. **Noise Injection**: We inject a small amount of Gaussian noise (epsilon=0.15).
3. **Perturbed Extraction**: We compute the saliency map for the noisy image.
4. **Comparison**: We use Structural Similarity Index (SSIM) and Correlation Coefficient (CC) to measure the stability of the explanation.

In [ ]:
# Load data and dummy-train
transform = transforms.Compose([transforms.ToTensor()])
train_dataset = datasets.MNIST('../data', train=True, download=True, transform=transform)
train_loader = torch.utils.data.DataLoader(torch.utils.data.Subset(train_dataset, range(1000)), batch_size=32, shuffle=True)
model = SimpleCNN().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
for data, target in train_loader:
    optimizer.zero_grad()
    F.cross_entropy(model(data.to(device)), target.to(device)).backward()
    optimizer.step()

# Select a test sample
img, label = train_dataset[10]

# Original Saliency
s_orig = get_saliency(model, img, label)

# Add Noise (epsilon = 0.15)
noise = torch.randn_like(img) * 0.15
img_noisy = torch.clamp(img + noise, 0, 1)
s_noisy = get_saliency(model, img_noisy, label)

# Verify that predictions didn't change
with torch.no_grad():
    pred_orig = model(img.unsqueeze(0).to(device)).argmax().item()
    pred_noisy = model(img_noisy.unsqueeze(0).to(device)).argmax().item()

# Calculate Robustness Metrics
cc = mc.linear_correlation_coefficient(s_orig, s_noisy)
ssim = mc.ssim(s_orig, s_noisy)
mse = mc.mean_squared_error(s_orig, s_noisy)

## 4. Results and Findings
**Findings:**
- Even though the added noise is minor, and the model's top-1 prediction **remains completely unchanged**, the explanation map shifts significantly.
- The SSIM (Structural Dissimilarity) score highlights how the perceptual structure of the saliency map fractures, moving from a localized focus to a noisy, dispersed pattern.
- This proves that relying solely on Top-1 Accuracy for model security is insufficient. `saliencytools` provides the necessary metrics to audit the underlying robustness of the model's reasoning process.

In [ ]:
print("--- Robustness Metrics (Original vs Noisy) ---")
print(f"Prediction Original: {pred_orig} | Prediction Noisy: {pred_noisy}")
print(f"Correlation (CC): {cc:.4f}")
print(f"Structural Dissimilarity (SSIM): {ssim:.4f}")

plt.figure(figsize=(15, 5))
plt.subplot(1, 4, 1); plt.title("Original Image"); plt.imshow(img.squeeze(), cmap='gray')
plt.subplot(1, 4, 2); plt.title("Noisy Image (eps=0.15)"); plt.imshow(img_noisy.squeeze(), cmap='gray')
plt.subplot(1, 4, 3); plt.title("Saliency (Original)"); plt.imshow(s_orig, cmap='hot')
plt.subplot(1, 4, 4); plt.title(f"Saliency (Noisy - CC:{cc:.2f})"); plt.imshow(s_noisy, cmap='hot')
plt.tight_layout()
plt.show()